# Degradation Risk Classifier — Training Notebook

This notebook trains the Random Forest model that powers the **Degradation risk** tab of the SoilSense dashboard. It produces `models/degradation_rf.joblib`, which the Streamlit app loads at runtime.

## Approach

For this portfolio demonstration we train on a **synthetic dataset generated by a rule-based *teacher* function** that encodes well-established degradation signals. This has three advantages:

1. **Transparency** — the teacher is a short, auditable Python function.
2. **Reproducibility** — no external data dependencies; anyone can run this.
3. **Honesty** — I'm not claiming ground-truth-backed performance I can't demonstrate.

In a production setting you would replace the labels with one of:
- LADA-L (Land Degradation Assessment in Drylands) polygons
- ESA CCI land-cover change (multi-epoch degradation proxy)
- [Trends.Earth](https://trends.earth) SDG 15.3.1 sub-indicator layers
- In-situ survey data from FAO / UNCCD partners

## Features

| Feature | Source | Units |
|---|---|---|
| soc | SoilGrids 2.0 | g/kg |
| phh2o | SoilGrids 2.0 | – |
| clay, sand | SoilGrids 2.0 | % |
| nitrogen | SoilGrids 2.0 | g/kg |
| cec | SoilGrids 2.0 | cmol(+)/kg |
| bdod | SoilGrids 2.0 | g/cm³ |
| rainfall_mm | CHIRPS annual sum | mm |
| slope_pct | SRTM derived | % |
| ndvi_mean | MODIS 3-yr mean | – |
| ndvi_trend | MODIS linear slope (per day) | – |

In [ ]:
import sys
sys.path.insert(0, '..')

from src.ml_model import (
    generate_synthetic_training_set,
    train_and_save_model,
    FEATURE_COLUMNS,
    MODEL_PATH,
)

import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 110

## 1. Generate the training set

In [ ]:
df = generate_synthetic_training_set(n=5000, seed=42)
print(f'Rows: {len(df)}  |  Degraded share: {df.label.mean():.1%}')
df.head()

In [ ]:
df.describe().round(2)

### Label distribution across key drivers

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, col in zip(axes.flat, ['soc', 'ndvi_trend', 'rainfall_mm', 'slope_pct']):
    for label, grp in df.groupby('label'):
        ax.hist(grp[col], bins=30, alpha=0.55,
                label=f'{"Degraded" if label else "Stable"} (n={len(grp)})')
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2. Train and evaluate

In [ ]:
metrics = train_and_save_model()
print(f'Model saved to: {MODEL_PATH}')
print('Held-out metrics:')
for k, v in metrics.items():
    print(f'  {k:>12}: {v}')

## 3. Feature importance

In [ ]:
import joblib
bundle = joblib.load(MODEL_PATH)
importances = pd.Series(bundle['model'].feature_importances_,
                        index=bundle['features']).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', color='#A0522D', ax=ax)
ax.set_xlabel('Feature importance')
ax.set_title('Degradation classifier — feature importance')
plt.tight_layout()
plt.show()

importances

## 4. Next steps to productionize

1. Replace the synthetic teacher labels with real degradation polygons (Trends.Earth, LADA-L, or project M&E data).
2. Retrain on geographically stratified samples to avoid spatial leakage — use `GroupKFold` on admin units.
3. Add uncertainty quantification (quantile forests or conformal prediction) and surface it in the UI.
4. Benchmark against a simple NDVI-trend-only baseline — any ML model should clearly beat it.